# Lab 02 .

In [1]:
import os, re, csv, math, statistics
from collections import Counter, defaultdict
from pprint import pprint
import pandas as pd

PDF_PATH = None                  # set to a multi-page PDF path
DOCX_PATH = "/mnt/data/Lab 02 Task.docx"
DOCX_A_PATH = None               # set to Document A
DOCX_B_PATH = None               # set to Document B

def pdf_pages(path):
    if not path or not os.path.exists(path):
        raise FileNotFoundError("Set PDF_PATH to an existing multi-page PDF.")
    try:
        from pypdf import PdfReader
    except ImportError:
        from PyPDF2 import PdfReader
    return [p.extract_text() or "" for p in PdfReader(path).pages]

def get_doc(path):
    from docx import Document
    if not path or not os.path.exists(path):
        raise FileNotFoundError("Set the DOCX path to an existing .docx file.")
    return Document(path)

def get_json(url):
    import requests
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        raise ConnectionError(f"{url} returned HTTP {r.status_code}; expected 200")
    data = r.json()
    if not isinstance(data, list):
        raise TypeError(f"{url} did not return a JSON list")
    return data

def word_tokens(text):
    return re.findall(r"\b[a-z]+(?:'[a-z]+)?\b", text.lower())

print("Setup ready.")

Setup ready.


## Task 1 — Multi-Page PDF Extraction with Word Frequency Analysis

In [2]:
try:
    pages = pdf_pages(PDF_PATH)
    print("Pages extracted:", len(pages))
    merged = "\n".join(pages)
    normalized = re.sub(r"[\W\d_]+", " ", merged).lower()
    tokens = normalized.split()
    print("Token count:", len(tokens))
    stopwords20 = set("a an the and or but if while is am are was were be been being do does did have has had of to in on at by for from with about as into through during before after above below up down out off over under again further then once here there when where why how all any both each few more most other some such no nor not only own same so than too very can will just should now".split())
    frequencies = Counter(t for t in tokens if t not in stopwords20)
    print("Top 10 words:")
    for word, count in frequencies.most_common(10): print(f"{word}: {count}")
    keyword = input("Keyword to search in PDF: ").strip().lower()
    found = []
    for page_no, text in enumerate(pages, 1):
        for sentence in re.split(r"(?<=[.!?])\s+", text):
            if keyword and keyword in sentence.lower(): found.append((page_no, sentence.strip()))
    if found:
        for page_no, sentence in found: print(f"Page {page_no}: {sentence}")
    else: print(f"No sentence containing {keyword!r} was found.")
except FileNotFoundError as e:
    print(e)

Set PDF_PATH to an existing multi-page PDF.


## Task 2 — Structured Page Metadata Extraction and CSV Export

In [3]:
try:
    pages = pdf_pages(PDF_PATH)
    pages_data = [{"page_number": i, "raw_text": text,
                   "word_count": len(re.findall(r"\b\w+\b", text)),
                   "char_count": len(re.sub(r"\s", "", text))}
                  for i, text in enumerate(pages, 1)]
    print("First two entries:"); pprint(pages_data[:2])
    if pages_data:
        high = max(pages_data, key=lambda x: x["word_count"])
        nonzero = [p for p in pages_data if p["word_count"] > 0]
        print("Highest word-count page:", high["page_number"], high["word_count"], "\nFull content:\n", high["raw_text"])
        print("Lowest non-zero page:", min(nonzero, key=lambda x:x["word_count"]) if nonzero else "No non-zero pages")
        print("Average words/page:", round(statistics.mean(p["word_count"] for p in pages_data), 2))
    class PhraseNotFoundError(Exception): pass
    phrase = input("Phrase to search: ").strip()
    hits = [(p["page_number"], p["raw_text"].lower().count(phrase.lower())) for p in pages_data if phrase and phrase.lower() in p["raw_text"].lower()]
    try:
        if not hits: raise PhraseNotFoundError(f"Phrase {phrase!r} not found")
        for page_no, count in hits: print(f"Page {page_no}: {count} occurrence(s)")
    except PhraseNotFoundError as e: print(e)
    with open("pdf_summary.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["page_number","word_count","char_count"])
        w.writerows([[p["page_number"],p["word_count"],p["char_count"]] for p in pages_data])
    with open("pdf_summary.csv", newline="", encoding="utf-8") as f:
        print("CSV rows:")
        for row in csv.reader(f): print(row)
except FileNotFoundError as e:
    print(e)

Set PDF_PATH to an existing multi-page PDF.


## Task 3 — PDF Text Preprocessing Pipeline for NLP

In [4]:
try:
    pages = pdf_pages(PDF_PATH)
    raw = "\n".join(pages)
    print("Characters before:", len(raw))
    cleaned = re.sub(r"https?://\S+|www\.\S+", " ", raw)
    cleaned = re.sub(r"\d+", " ", cleaned)
    cleaned = re.sub(r"[^A-Za-z. ]", " ", cleaned)
    cleaned = re.sub(r" {2,}", " ", cleaned).lower()
    print("Characters after:", len(cleaned))
    pieces = [s.strip() for s in re.split(r"[.!?]", cleaned)]
    print("Sentence pieces before removing empties:", len(pieces))
    sentences = [s for s in pieces if s]
    print("Sentence count after removing empties:", len(sentences))
    import nltk
    try: from nltk.corpus import stopwords; sw = set(stopwords.words("english"))
    except LookupError:
        nltk.download("stopwords", quiet=True)
        from nltk.corpus import stopwords; sw = set(stopwords.words("english"))
    except Exception: sw = set("a an the and or is are was were to of in on for with".split())
    filtered = [" ".join(w for w in word_tokens(s) if w not in sw) for s in sentences]
    kept = [s for s in filtered if len(s.split()) >= 5]
    print("Before >=5-token filter:", len(filtered), "| After:", len(kept))
    print("Samples:", kept[:3])
    with open("nlp_ready_output.txt","w",encoding="utf-8") as f: f.write("\n".join(kept))
    with open("nlp_ready_output.txt",encoding="utf-8") as f: lines = [x.rstrip("\n") for x in f]
    print("Lines written:", len(lines))
    if lines:
        print("Longest:", max(lines,key=len))
        print("Shortest:", min(lines,key=len))
    else: print("No sentences passed the filter.")
except FileNotFoundError as e: print(e)

Set PDF_PATH to an existing multi-page PDF.


## Task 4 — DOCX Paragraph Classification and Statistical Analysis

In [5]:
try:
    doc = get_doc(DOCX_PATH)
    paras = [{"text": p.text, "style": p.style.name if p.style else "Unknown"} for p in doc.paragraphs]
    print("Paragraph count:", len(paras))
    print("Unique styles:", sorted({p["style"] for p in paras}))
    groups = {"Headings": [], "Body": [], "Other/Empty": []}
    for p in paras:
        s = p["style"]
        if s.startswith("Heading"): groups["Headings"].append(p)
        elif s == "Normal" or "Body" in s: groups["Body"].append(p)
        else: groups["Other/Empty"].append(p)
    print("Group counts:", {k:len(v) for k,v in groups.items()})
    qualifying = [{"text":p["text"],"word_count":len(p["text"].split()),
                   "char_count":len(re.sub(r"\s","",p["text"]))}
                  for p in groups["Body"] if len(p["text"].split()) > 5]
    if qualifying:
        print("Highest:", max(qualifying,key=lambda x:x["word_count"]))
        print("Lowest >5 words:", min(qualifying,key=lambda x:x["word_count"]))
        print("Mean word count:", round(statistics.mean(x["word_count"] for x in qualifying),2))
    else: print("No qualifying Body paragraphs.")
    nested = {}
    for p in paras:
        nested.setdefault(p["style"], {"count":0,"texts":[]})
        nested[p["style"]]["count"] += 1; nested[p["style"]]["texts"].append(p["text"])
    for info in nested.values():
        if info["count"] > 2: info["texts"] = info["texts"][:2]
    nested = dict(sorted(nested.items(), key=lambda kv:kv[1]["count"], reverse=True))
    pprint(nested, sort_dicts=False)
except Exception as e: print("DOCX error:", e)

DOCX error: Set the DOCX path to an existing .docx file.


## Task 5 — DOCX Table Extraction and Pandas Integration

In [6]:
try:
    from docx.text.paragraph import Paragraph
    from docx.oxml.table import CT_Tbl
    doc = get_doc(DOCX_PATH)
    tables = [[[cell.text for cell in row.cells] for row in t.rows] for t in doc.tables]
    print("Tables found:", len(tables))
    for i,t in enumerate(tables,1): print(f"Table {i}: {len(t)} rows × {max(map(len,t), default=0)} columns")
    if tables and tables[0]:
        first = tables[0]
        header = [str(x).strip() or f"column_{i+1}" for i,x in enumerate(first[0])]
        data = [(r + [""]*len(header))[:len(header)] for r in first[1:]]
        df = pd.DataFrame(data, columns=header)
        display(df)
        print("Empty/None cells per column:", {c:int(df[c].isna().sum()+(df[c].astype(str).str.strip()=="").sum()) for c in df.columns})
    else:
        df = pd.DataFrame(); print("No table found; set DOCX_PATH to a DOCX containing a table.")
    before_table = []
    for child in doc.element.body.iterchildren():
        if isinstance(child, CT_Tbl): break
        if child.tag.endswith("}p"): before_table.append(Paragraph(child,doc).text)
    print("Pre-table paragraph count:", len(before_table))
    with open("docx_extracted.txt","w",encoding="utf-8") as f:
        f.write("--- PRE-TABLE PARAGRAPHS ---\n")
        for p in before_table: f.write(p+"\n")
        f.write("\n--- TABLE DATA ---\n")
        if tables:
            for row in tables[0]: f.write(" | ".join(row)+"\n")
    print("Saved docx_extracted.txt")
except Exception as e: print("DOCX/table error:", e)

DOCX/table error: Set the DOCX path to an existing .docx file.


## Task 6 — Multi-Document Comparison, Deduplication, and Frequency Report

In [7]:
try:
    a, b = get_doc(DOCX_A_PATH), get_doc(DOCX_B_PATH)
    headings_a = [p.text.strip() for p in a.paragraphs if p.style and p.style.name.startswith("Heading")]
    headings_b = [p.text.strip() for p in b.paragraphs if p.style and p.style.name.startswith("Heading")]
    print("Common headings:", sorted(set(headings_a)&set(headings_b)))
    print("Only Document A:", sorted(set(headings_a)-set(headings_b)))
    print("Only Document B:", sorted(set(headings_b)-set(headings_a)))
    body_a = [p.text for p in a.paragraphs if p.style and p.style.name=="Normal"]
    body_b = [p.text for p in b.paragraphs if p.style and p.style.name=="Normal"]
    merged = body_a + body_b; unique=[]; seen=set()
    for p in merged:
        key=p.strip().lower()
        if key not in seen: seen.add(key); unique.append(p.strip())
    print("Body paragraphs before/after deduplication:", len(merged), len(unique))
    sw = set("a an the and or but if is are was were be been being to of in on for from with at by as it this that these those".split())
    all_tokens=[]
    for p in unique:
        all_tokens += [w for w in re.sub(r"[^a-z\s]"," ",p.lower()).split() if w not in sw]
    counter=Counter(all_tokens)
    print("Top 20 words:", counter.most_common(20))
    lens=[len(word_tokens(p)) for p in unique]
    print("Total headings:",len(headings_a)+len(headings_b))
    print("Unique body paragraphs:",len(unique))
    print("Vocabulary size after stop-word removal:",len(set(all_tokens)))
    print("Average paragraph length:",round(statistics.mean(lens),2) if lens else 0)
    if unique:
        idx=max(range(len(unique)),key=lambda i:lens[i])
        print("Longest paragraph:",unique[idx], "| words:",lens[idx])
except Exception as e:
    print("Set DOCX_A_PATH and DOCX_B_PATH to two existing DOCX files:", e)

Set DOCX_A_PATH and DOCX_B_PATH to two existing DOCX files: Set the DOCX path to an existing .docx file.


## Task 7 — Multi-Endpoint API Data Joining and Analysis

In [8]:
try:
    users = get_json("https://jsonplaceholder.typicode.com/users")
    posts = get_json("https://jsonplaceholder.typicode.com/posts")
    posts_by_user=defaultdict(list)
    for p in posts: posts_by_user[p["userId"]].append(p)
    joined={}
    for u in users:
        ps=posts_by_user[u["id"]]
        joined[u["name"]]={"email":u["email"],"post_count":len(ps),"titles":[p["title"] for p in ps]}
    print("First 3 users:"); pprint(dict(list(joined.items())[:3]))
    top=max(users,key=lambda u:len(posts_by_user[u["id"]]))
    print("Most posts — name:",top["name"],"| email:",top["email"],"| company:",top["company"]["name"])
    for i,p in enumerate(posts_by_user[top["id"]],1): print(f"{i}. {p['title']}")
    print("Filtered users (synergies in company.bs OR city length > 8):")
    for u in users:
        city=u.get("address",{}).get("city","")
        bs=u.get("company",{}).get("bs","")
        if "synergies" in bs.lower() or len(city)>8:
            print(u["name"],"|",city,"| post count:",len(posts_by_user[u["id"]]))
except Exception as e: print("API task failed:",e)

First 3 users:
{'Clementine Bauch': {'email': 'Nathan@yesenia.net',
                      'post_count': 10,
                      'titles': ['asperiores ea ipsam voluptatibus modi minima '
                                 'quia sint',
                                 'dolor sint quo a velit explicabo quia nam',
                                 'maxime id vitae nihil numquam',
                                 'autem hic labore sunt dolores incidunt',
                                 'rem alias distinctio quo quis',
                                 'est et quae odit qui non',
                                 'quasi id et eos tenetur aut quo autem',
                                 'delectus ullam et corporis nulla voluptas '
                                 'sequi',
                                 'iusto eius quod necessitatibus culpa ea',
                                 'a quo magni similique perferendis']},
 'Ervin Howell': {'email': 'Shanna@melissa.tv',
                  'post_count

## Task 8 — To-Do Completion Rate Analysis Across Users

In [9]:
try:
    todos=get_json("https://jsonplaceholder.typicode.com/todos")
    if not todos: raise ValueError("Todo endpoint returned an empty list.")
    print("Todo records:",len(todos))
    stats={}
    for t in todos:
        uid=t["userId"]; stats.setdefault(uid,{"total":0,"completed":0,"incomplete":0})
        stats[uid]["total"]+=1
        stats[uid]["completed" if t["completed"] else "incomplete"]+=1
    pprint(stats)
    for uid,s in sorted(stats.items()):
        print(f"User {uid}: completed + incomplete == total: {s['completed']+s['incomplete']==s['total']}")
    rates={uid:round(s["completed"]/s["total"]*100,1) for uid,s in stats.items() if s["total"]}
    hi=max(rates,key=rates.get); lo=min(rates,key=rates.get)
    print("Highest rate:",hi,rates[hi],"%")
    print("Lowest rate:",lo,rates[lo],"%")
    print("Average user completion rate:",round(statistics.mean(rates.values()),1),"%")
    users=get_json("https://jsonplaceholder.typicode.com/users")
    lookup={u["id"]:u["name"] for u in users}
    rows=sorted([(rates[uid],lookup.get(uid,f"User {uid}"),s["total"],s["completed"],uid) for uid,s in stats.items()],reverse=True)
    header=f"{'Rank':<5} | {'Name':<25} | {'Total Tasks':>11} | {'Completed':>9} | {'Completion Rate %':>17}"
    print(header); print("-"*len(header))
    for rank,(rate,name,total,done,uid) in enumerate(rows,1):
        print(f"{rank:<5} | {name:<25.25} | {total:>11} | {done:>9} | {rate:>17.1f}")
except Exception as e: print("Todo/API validation error:",e)

Todo records: 200
{1: {'completed': 11, 'incomplete': 9, 'total': 20},
 2: {'completed': 8, 'incomplete': 12, 'total': 20},
 3: {'completed': 7, 'incomplete': 13, 'total': 20},
 4: {'completed': 6, 'incomplete': 14, 'total': 20},
 5: {'completed': 12, 'incomplete': 8, 'total': 20},
 6: {'completed': 6, 'incomplete': 14, 'total': 20},
 7: {'completed': 9, 'incomplete': 11, 'total': 20},
 8: {'completed': 11, 'incomplete': 9, 'total': 20},
 9: {'completed': 8, 'incomplete': 12, 'total': 20},
 10: {'completed': 12, 'incomplete': 8, 'total': 20}}
User 1: completed + incomplete == total: True
User 2: completed + incomplete == total: True
User 3: completed + incomplete == total: True
User 4: completed + incomplete == total: True
User 5: completed + incomplete == total: True
User 6: completed + incomplete == total: True
User 7: completed + incomplete == total: True
User 8: completed + incomplete == total: True
User 9: completed + incomplete == total: True
User 10: completed + incomplete == to

## Task 9 — Nested JSON Flattening, DataFrame Operations, and Comment Analysis

In [10]:
try:
    users=get_json("https://jsonplaceholder.typicode.com/users")
    def flatten_user(user:dict)->dict:
        address=user.get("address",{}); geo=address.get("geo",{}); company=user.get("company",{})
        return {"id":user.get("id"),"name":user.get("name"),"username":user.get("username"),
                "email":user.get("email"),"city":address.get("city"),"zipcode":address.get("zipcode"),
                "lat":geo.get("lat"),"lng":geo.get("lng"),"company_name":company.get("name"),
                "company_catchphrase":company.get("catchPhrase")}
    flattened=[flatten_user(u) for u in users]
    pprint(flattened)
    df=pd.DataFrame(flattened)
    df["lat"]=df["lat"].astype(float); df["lng"]=df["lng"].astype(float)
    df["distance_from_origin"]=df.apply(lambda r:math.sqrt(r["lat"]**2+r["lng"]**2),axis=1)
    display(df)
    print("Farthest user:"); display(df.loc[[df["distance_from_origin"].idxmax()]])
    north=df[df["lat"]>0]; south=df[df["lat"]<0]
    print("Northern Hemisphere:",north["name"].tolist())
    print("Southern Hemisphere:",south["name"].tolist())
    navg=north["distance_from_origin"].mean(); savg=south["distance_from_origin"].mean()
    print("Northern average distance:",round(navg,4) if pd.notna(navg) else "N/A")
    print("Southern average distance:",round(savg,4) if pd.notna(savg) else "N/A")
    if pd.isna(navg) or pd.isna(savg): print("Cannot compare; one group is empty.")
    else: print("Higher average:", "Northern" if navg>savg else "Southern" if savg>navg else "Equal")
    posts=get_json("https://jsonplaceholder.typicode.com/posts")
    comments=get_json("https://jsonplaceholder.typicode.com/comments")
    counts=Counter(c["postId"] for c in comments)
    tuples=[(p["id"],p["title"][:40],counts.get(p["id"],0)) for p in posts]
    print("Top 5 most-commented:",sorted(tuples,key=lambda x:(-x[2],x[0]))[:5])
    print("Bottom 5 least-commented:",sorted(tuples,key=lambda x:(x[2],x[0]))[:5])
except Exception as e: print("API/dataframe task failed:",e)

[{'city': 'Gwenborough',
  'company_catchphrase': 'Multi-layered client-server neural-net',
  'company_name': 'Romaguera-Crona',
  'email': 'Sincere@april.biz',
  'id': 1,
  'lat': '-37.3159',
  'lng': '81.1496',
  'name': 'Leanne Graham',
  'username': 'Bret',
  'zipcode': '92998-3874'},
 {'city': 'Wisokyburgh',
  'company_catchphrase': 'Proactive didactic contingency',
  'company_name': 'Deckow-Crist',
  'email': 'Shanna@melissa.tv',
  'id': 2,
  'lat': '-43.9509',
  'lng': '-34.4618',
  'name': 'Ervin Howell',
  'username': 'Antonette',
  'zipcode': '90566-7771'},
 {'city': 'McKenziehaven',
  'company_catchphrase': 'Face to face bifurcated interface',
  'company_name': 'Romaguera-Jacobson',
  'email': 'Nathan@yesenia.net',
  'id': 3,
  'lat': '-68.6102',
  'lng': '-47.0653',
  'name': 'Clementine Bauch',
  'username': 'Samantha',
  'zipcode': '59590-4157'},
 {'city': 'South Elvis',
  'company_catchphrase': 'Multi-tiered zero tolerance productivity',
  'company_name': 'Robel-Corkery'

,id,name,username,email,city,zipcode,lat,lng,company_name,company_catchphrase,distance_from_origin
0,1,Leanne Graham,Bret,Sincere@april.biz,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,89.318161
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,55.850669
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,83.201575
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,166.918807
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,70.161149
5,6,Mrs. Dennis Schulist,Leopoldo_Corkery,Karley_Dach@jasper.info,South Christy,23505-1337,-71.4197,71.7478,Considine-Lockman,Synchronised bottom-line interface,101.234976
6,7,Kurtis Weissnat,Elwyn.Skiles,Telly.Hoeger@billy.biz,Howemouth,58804-1099,24.8918,21.8984,Johns Group,Configurable multimedia task-force,33.153305
7,8,Nicholas Runolfsdottir V,Maxime_Nienow,Sherwood@rosamond.me,Aliyaview,45169,-14.3990,-120.7677,Abernathy Group,Implemented secondary concept,121.623059
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,Switchable contextually-based project,170.677768
9,10,Clementina DuBuque,Moriah.Stanton,Rey.Padberg@karina.biz,Lebsackbury,31428-2261,-38.2386,57.2232,Hoeger LLC,Centralized empowering task-force,68.823580


Farthest user:


,id,name,username,email,city,zipcode,lat,lng,company_name,company_catchphrase,distance_from_origin
8,9,Glenna Reichert,Delphine,Chaim_McDermott@dana.io,Bartholomebury,76495-3109,24.6463,-168.8889,Yost and Sons,Switchable contextually-based project,170.677768


Northern Hemisphere: ['Patricia Lebsack', 'Kurtis Weissnat', 'Glenna Reichert']
Southern Hemisphere: ['Leanne Graham', 'Ervin Howell', 'Clementine Bauch', 'Chelsey Dietrich', 'Mrs. Dennis Schulist', 'Nicholas Runolfsdottir V', 'Clementina DuBuque']
Northern average distance: 123.5833
Southern average distance: 84.3162
Higher average: Northern
Top 5 most-commented: [(1, 'sunt aut facere repellat provident occae', 5), (2, 'qui est esse', 5), (3, 'ea molestias quasi exercitationem repell', 5), (4, 'eum et est occaecati', 5), (5, 'nesciunt quas odio', 5)]
Bottom 5 least-commented: [(1, 'sunt aut facere repellat provident occae', 5), (2, 'qui est esse', 5), (3, 'ea molestias quasi exercitationem repell', 5), (4, 'eum et est occaecati', 5), (5, 'nesciunt quas odio', 5)]


## Before submitting
- Set `PDF_PATH` to a real multi-page PDF for Tasks 1–3.
- Set `DOCX_PATH` to a DOCX containing a table for Task 5. The uploaded assignment DOCX may not contain a table.
- Set `DOCX_A_PATH` and `DOCX_B_PATH` to two different documents for Task 6.
- Tasks 7–9 call JSONPlaceholder and require internet access.